In [1]:
!pip install groq python-dotenv

In [2]:
import os
from dotenv import load_dotenv
from groq import Groq

In [4]:
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API Key: ")

client = Groq(api_key=GROQ_API_KEY)

MODEL_NAME = "llama-3.3-70b-versatile"

print("Groq client initialized successfully.")

Enter your Groq API Key:  ········


Groq client initialized successfully.


In [5]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": (
            "You are a Technical Support Expert.\n"
            "Be precise, logical, and code-focused.\n"
            "Provide debugging steps and corrected code.\n"
            "Avoid unnecessary conversation."
        ),
        "temperature": 0.7
    },
    "billing": {
        "system_prompt": (
            "You are a Billing Support Specialist.\n"
            "Be empathetic, professional, and policy-driven.\n"
            "Help users understand charges, refunds, and subscriptions clearly."
        ),
        "temperature": 0.7
    },
    "general": {
        "system_prompt": (
            "You are a friendly assistant.\n"
            "Engage naturally and conversationally."
        ),
        "temperature": 0.7
    }
}

In [10]:
def route_prompt(user_input: str) -> str:
    """
    Classifies user intent into:
    technical | billing | general | tool
    Returns ONLY the category name.
    """

    routing_prompt = f"""
You are an intent classifier for a customer support AI system.

Classify the user message into one of these categories:

technical → coding issues, bugs, errors, debugging
billing → payments, charges, refunds, subscriptions
tool → requests for real-time data like cryptocurrency prices, stock prices, weather, or live information
general → casual conversation or anything else

Return ONLY one word from:
technical
billing
tool
general

User message:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a strict classifier."},
            {"role": "user", "content": routing_prompt}
        ],
        temperature=0
    )

    category = response.choices[0].message.content.strip().lower()

    allowed = ["technical", "billing", "general", "tool"]
    if category not in allowed:
        return "general"

    return category

In [11]:
def get_bitcoin_price():
    """
    Simulates fetching Bitcoin price.
    Replace with real API call in production.
    """
    return "$51,250 USD"


def tool_handler(user_input: str) -> str:
    if "bitcoin" in user_input.lower():
        price = get_bitcoin_price()
        return f"The current price of Bitcoin is {price}."
    
    return "Tool requested but no matching tool found."

In [12]:
def process_request(user_input: str) -> str:
    """
    Full Mixture of Experts Flow:
    1. Route intent
    2. Call tool if needed
    3. Otherwise call appropriate expert
    """

    category = route_prompt(user_input)
    print(f"[Router Decision]: {category}")

    if category == "tool":
        return tool_handler(user_input)

    expert_config = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input}
        ],
        temperature=expert_config["temperature"]
    )

    return response.choices[0].message.content

In [ ]:
while True:
    user_query = input("\nUser: ")

    if user_query.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    answer = process_request(user_query)
    print(f"\nAssistant:\n{answer}")


User:  what is the bitcoin price today?


[Router Decision]: tool

Assistant:
The current price of Bitcoin is $51,250 USD.



User:  will it rain today?


[Router Decision]: tool

Assistant:
Tool requested but no matching tool found.



User:  what inbuilt function exists in python that allows use of ML techniques?


[Router Decision]: technical

Assistant:
**Python Inbuilt Functions for Machine Learning:**

Python has several libraries that provide machine learning (ML) capabilities. While there are no built-in ML functions in the Python standard library, the following libraries can be used for ML tasks:

1. **scikit-learn**: A widely used library for ML tasks, including classification, regression, clustering, and more.
2. **TensorFlow**: An open-source library for deep learning tasks, including neural networks and convolutional neural networks.
3. **PyTorch**: Another popular library for deep learning tasks, known for its ease of use and flexibility.
4. **Keras**: A high-level neural networks API, capable of running on top of TensorFlow, PyTorch, or Theano.

Some of the key functions in these libraries include:

* **scikit-learn:**
	+ `train_test_split` for splitting data into training and testing sets
	+ `LinearRegression` for linear regression tasks
	+ `DecisionTreeClassifier` for decision tree